# ML-KEM-1024 板上验证

点击 **Kernel → Restart & Run All**，确认后即可完成加载、三例官方测试、结果显示和 JSON 保存。无需复制命令或打开终端。

默认测试 KeyGen、Encaps、Decaps 三例。需要完整 145 条时，将下面的 `RUN_FULL_145` 改为 `True`，再运行全部单元。


## 1. 配置与加载

校验 bitstream、HWH、驱动和向量，再加载硬件。Notebook 可以正常保存；模板哈希仅作来源记录。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from mlkem1024 import MLKEM1024

RUN_FULL_145 = False
BUNDLE = Path.cwd()
board = MLKEM1024.load(BUNDLE / 'release_manifest.json')
print('硬件加载成功；官方向量：', len(board.vectors['cases']))

def save_result(report, label):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = board.export(report, BUNDLE / 'evidence' / ('%s_%s.json' % (stamp, label)))
    print('结果已保存：', path)
    return path


## 2. 三例验证

覆盖索引 0、1、115。逐字节核对官方输出、返回值和硬件调用计数；完成后立即保存结果。


In [ ]:
subset = board.run_cases([0, 1, 115], timeout=20.0)
subset_path = save_result(subset, 'subset')
print(subset['status'], subset['unique_checked_cases'], '/ 145')
for case in subset['cases']:
    print(case['case_index'], case['operation'], 'PASS' if case['passed'] else case['error'])


## 3. 完整测试（默认关闭）

需要全量时只需修改顶部 `RUN_FULL_145` 开关。只有 145 个不同索引全部通过才会显示 `FULL_PASS`。


In [ ]:
full = None
full_path = None
if RUN_FULL_145 and subset['passed']:
    full = board.run_cases(range(145), timeout=20.0)
    full_path = save_result(full, 'full')
    print(full['status'], full['unique_checked_cases'], '/ 145')
elif RUN_FULL_145:
    print('三例测试失败，停止全量测试；失败证据已保存。')
else:
    print('完整 145 条测试未启用。')


## 4. API cycles and transport/wall time
**Live API time** is the firmware's raw `rdcycle` interval divided by 100 MHz. The empty bracket is also reported separately; API time excludes PS upload/download and host verification. **Transport** is measured upload plus download time; **run wait** includes CPU startup and host polling; **wall time** covers the whole host transaction. Operation 4 includes seed KeyGen followed by Decaps.

The comparison column is a **historical RTL CPU-only baseline** from the frozen `rv32im_fast` results. It is not a live CPU-only board measurement. No measured board speedup is claimed until both the CPU-only and accelerated overlays are measured on the board under matching conditions.

In [ ]:
from html import escape
from IPython.display import HTML, display

report = full if full is not None else subset
columns = ['Index', 'Operation', 'Live API cycles', 'Live API ms', 'API minus empty ms',
           'Transport ms', 'Run wait ms', 'Host wall ms', 'Historical RTL CPU-only cycles']
rows = []
for case in report['cases']:
    if not case['passed']:
        print('FAILED:', case['case_index'], case['error_type'], case['error'])
        continue
    rows.append([case['case_index'], case['operation'], case['raw_cycles'],
                 f"{case['algorithm_seconds'] * 1000:.3f}",
                 f"{case['algorithm_seconds_minus_empty'] * 1000:.3f}",
                 f"{(case['upload_seconds'] + case['download_seconds']) * 1000:.3f}",
                 f"{case['run_wait_seconds'] * 1000:.3f}",
                 f"{case['wall_seconds'] * 1000:.3f}",
                 case['historical_rtl_cpu_only']['raw_cycles']])
header = '<tr>' + ''.join('<th>' + escape(label) + '</th>' for label in columns) + '</tr>'
body = ''.join('<tr>' + ''.join('<td>' + escape(str(value)) + '</td>' for value in row) + '</tr>'
               for row in rows)
display(HTML('<table>' + header + body + '</table>'))
print('Report status:', report['status'], '| Total host wall seconds:', report['total_wall_seconds'])

## 5. 下载结果

以下链接可下载本次原始 JSON。每次运行使用独立文件名，保留通过和失败结果。


In [ ]:
from IPython.display import FileLink, display
for path in (subset_path, full_path):
    if path is not None:
        display(FileLink(str(path.relative_to(BUNDLE))))


三例通过表示子集验证完成，不等于全量 145 条通过。表中的 CPU-only 周期来自历史 RTL 仿真，不能作为实板对照加速比；使用官方公开向量也不等于获得认证。
